# Final Detector Selection — Adaptive Road Vision

Select the best of the **two models actually used in this project**: Detector 1 — YOLO11n and Detector 2 — YOLO11s. This notebook reads the saved comparison metrics, ranks valid candidates, and copies the selected checkpoint to a stable path for downstream notebooks.

**Order:** run the YOLO11n training/evaluation notebook, the YOLO11s training/evaluation notebook, then Notebook 20 (detector comparison), then this notebook.

Selection uses the saved test mAP@0.50:0.95 as the primary metric and mAP@0.50, precision, and recall as tie-breakers. This is a model-selection record, not a claim that test metrics generalize to every road scene.

In [ ]:
from pathlib import Path
import os
import json
import shutil
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path(r"C:\Users\ankit\Desktop\Adaptive_Road_Vision")
if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(
        f"Project root not found: {PROJECT_ROOT}. Edit PROJECT_ROOT to your actual project folder."
    )
os.chdir(PROJECT_ROOT)

RESULT_ROOT = PROJECT_ROOT / "results" / "detection"
MODEL_ROOT = PROJECT_ROOT / "models" / "detection"
RESULT_ROOT.mkdir(parents=True, exist_ok=True)
MODEL_ROOT.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Results:", RESULT_ROOT)
print("Models:", MODEL_ROOT)

## 1. Load detector comparison results

In [ ]:
comparison_csv = RESULT_ROOT / "detector_comparison_summary.csv"
comparison_report_json = RESULT_ROOT / "detector_comparison_report.json"

if not comparison_csv.is_file():
    raise FileNotFoundError(
        f"Missing {comparison_csv}. Run Notebook 20 (YOLO11n vs YOLO11s comparison) first."
    )

comparison_df = pd.read_csv(comparison_csv)
required_columns = ["detector", "model", "precision", "recall", "mAP50", "mAP50-95"]
missing = [c for c in required_columns if c not in comparison_df.columns]
if missing:
    raise ValueError(f"Comparison CSV is missing required columns: {missing}")

# Only these two experiments are valid candidates for this project.
expected = {"Detector 1": "YOLO11n", "Detector 2": "YOLO11s"}
comparison_df = comparison_df[comparison_df["detector"].isin(expected)].copy()
if comparison_df.empty:
    raise ValueError("No Detector 1 / Detector 2 rows found. Regenerate the comparison using Notebook 20.")

for detector_name, model_name in expected.items():
    rows = comparison_df[comparison_df["detector"] == detector_name]
    if not rows.empty and not (rows["model"].astype(str) == model_name).all():
        raise ValueError(f"{detector_name} should be {model_name}; found {rows['model'].tolist()}")

# If duplicate rows exist, keep the latest single row per detector (normally there is one).
comparison_df = comparison_df.drop_duplicates(subset=["detector"], keep="last")
display(comparison_df)

In [ ]:
# Expected candidates only: Detector 1 — YOLO11n; Detector 2 — YOLO11s.
expected_detectors = {"Detector 1": "YOLO11n", "Detector 2": "YOLO11s"}
for detector_name, model_name in expected_detectors.items():
    rows = comparison_df[comparison_df["detector"] == detector_name]
    if rows.empty:
        print(f"WARNING: {detector_name} ({model_name}) is missing from the comparison CSV.")
    else:
        print(f"Found {detector_name}: {model_name}")
print("Candidate rows:", len(comparison_df))

## 2. Validate candidate metrics

Missing/non-numeric primary metrics are excluded from automatic selection. No metric is fabricated.


In [ ]:
metric_columns = ["precision", "recall", "mAP50", "mAP50-95"]
selection_df = comparison_df.copy()
for col in metric_columns:
    selection_df[col] = pd.to_numeric(selection_df[col], errors="coerce")

checkpoint_map = {
    "Detector 1": MODEL_ROOT / "detector_1_best.pt",
    "Detector 2": MODEL_ROOT / "detector_2_best.pt",
}
selection_df["checkpoint_path"] = selection_df["detector"].map(lambda d: str(checkpoint_map[d]))
selection_df["checkpoint_exists"] = selection_df["detector"].map(lambda d: checkpoint_map[d].is_file())
selection_df["metrics_valid"] = np.isfinite(selection_df["mAP50-95"])
selection_df["eligible"] = selection_df["metrics_valid"] & selection_df["checkpoint_exists"]

display(selection_df[["detector", "model"] + metric_columns + ["checkpoint_exists", "metrics_valid", "eligible"]])
valid_df = selection_df[selection_df["eligible"]].copy()
if valid_df.empty:
    raise ValueError(
        "No eligible detector has both a valid mAP50-95 metric and an existing checkpoint. "
        "Check results/detection/detector_comparison_summary.csv and models/detection/detector_1_best.pt / detector_2_best.pt."
    )

## 3. Apply the selection rule

Only YOLO11n and YOLO11s are considered. Candidates without a valid mAP@0.50:0.95 value or without their trained checkpoint are excluded. The highest mAP@0.50:0.95 wins; ties are broken by mAP@0.50, precision, then recall. Missing secondary metrics sort last.

In [ ]:
# Numeric secondary metrics; missing values rank below measured values.
for col in ["mAP50", "precision", "recall"]:
    valid_df[col] = pd.to_numeric(valid_df[col], errors="coerce")
ranked = valid_df.sort_values(
    by=["mAP50-95", "mAP50", "precision", "recall"],
    ascending=[False, False, False, False],
    na_position="last",
    kind="mergesort",
).reset_index(drop=True)

display(ranked[["detector", "model", "mAP50-95", "mAP50", "precision", "recall", "checkpoint_exists"]])
selected = ranked.iloc[0].to_dict()
selected_detector = str(selected["detector"])
selected_model = str(selected["model"])
selected_checkpoint = checkpoint_map[selected_detector]
print("Selected detector:", selected_detector)
print("Model:", selected_model)
print("Selection metric (test mAP@0.50:0.95):", selected["mAP50-95"])
print("Checkpoint:", selected_checkpoint)

## 4. Check checkpoint availability

In [ ]:
# Confirm the selected checkpoint before copying it for downstream use.
if selected_detector not in checkpoint_map:
    raise KeyError(f"No checkpoint mapping exists for {selected_detector}.")
selected_checkpoint = checkpoint_map[selected_detector]
print("Selected checkpoint:", selected_checkpoint)
print("Exists:", selected_checkpoint.is_file())
if not selected_checkpoint.is_file():
    raise FileNotFoundError(f"Selected trained checkpoint is missing: {selected_checkpoint}")

## 5. Create the stable downstream detector checkpoint

Later tracking and pipeline notebooks can use this stable path instead of hard-coding a detector-specific filename.


In [ ]:
stable_checkpoint = MODEL_ROOT / "final_selected_detector.pt"

# Avoid deleting the original detector checkpoint; create a copy for downstream use.
shutil.copy2(selected_checkpoint, stable_checkpoint)

print("Stable downstream checkpoint:", stable_checkpoint)


## 6. Selection visualization

In [ ]:
plot_metrics = [
    ("mAP50-95", "Test mAP@0.50:0.95"),
    ("mAP50", "Test mAP@0.50"),
    ("precision", "Test Precision"),
    ("recall", "Test Recall"),
]
for metric, title in plot_metrics:
    plt.figure(figsize=(7, 4))
    plot_df = selection_df.copy()
    plot_df[metric] = pd.to_numeric(plot_df[metric], errors="coerce")
    plt.bar(plot_df["model"], plot_df[metric])
    plt.ylabel(metric)
    plt.title(title)
    plt.tight_layout()
    plt.show()

## 7. Save the final detector-selection record

In [ ]:
selection_record = {
    "project_root": str(PROJECT_ROOT),
    "selection_rule": {
        "primary_metric": "test mAP50-95",
        "secondary_metrics": ["test mAP50", "test precision", "test recall"],
        "rule": "Among YOLO11n and YOLO11s with a valid mAP50-95 and an existing checkpoint, select highest mAP50-95; break ties by mAP50, precision, then recall. Missing secondary metrics rank last."
    },
    "selected_detector": selected_detector,
    "selected_model": selected_model,
    "selected_checkpoint": str(selected_checkpoint),
    "stable_downstream_checkpoint": str(stable_checkpoint),
    "selected_metrics": {
        metric: (float(selected[metric]) if pd.notna(selected.get(metric)) else None)
        for metric in ["mAP50-95", "mAP50", "precision", "recall"]
    },
    "all_candidates": [
        {k: (None if pd.isna(v) else (v.item() if isinstance(v, np.generic) else v))
         for k, v in row.items() if k not in ["checkpoint_path"]}
        for row in ranked[["detector", "model", "mAP50-95", "mAP50", "precision", "recall", "checkpoint_exists", "metrics_valid", "eligible"]].to_dict(orient="records")
    ],
}
selection_json = RESULT_ROOT / "final_detector_selection.json"
selection_json.write_text(json.dumps(selection_record, indent=2), encoding="utf-8")
selection_csv = RESULT_ROOT / "final_detector_selection_candidates.csv"
ranked[["detector", "model", "mAP50-95", "mAP50", "precision", "recall", "checkpoint_exists", "eligible"]].to_csv(selection_csv, index=False)
print("Saved:", selection_json)
print("Saved:", selection_csv)

## Final result

The selected model and measured metrics are saved to `results/detection/final_detector_selection.json`; candidate ranking is saved to `results/detection/final_detector_selection_candidates.csv`. The selected checkpoint is copied to `models/detection/final_selected_detector.pt` for downstream tracking and pipeline notebooks.

In [ ]:
print("=== FINAL DETECTOR SELECTION ===")
print("Detector:", selected_detector)
print("Model:", selected_model)
print("Test mAP50-95:", selected["mAP50-95"])
print("Test mAP50:", selected["mAP50"])
print("Test Precision:", selected["precision"])
print("Test Recall:", selected["recall"])
print("Checkpoint:", stable_checkpoint)
